# 서울 사고다발 지점 주변에는 교통시설이 몇 개 있을까?

2023~2025년 사고 분석에 사용한 **2025년 기준 사고누적지점**을 중심으로 주변 교통시설의 수를 센다. 결과표 한 행은 지점 하나이며, 사고 유형별 자료와 연결할 때 같은 지점에서 발생한 사고들은 같은 주변 시설 수를 공유한다.

위에서부터 셀을 하나씩 실행한다. 먼저 시설별 원본과 남길 열을 비교하고, 한 지점에서 카메라까지의 거리를 직접 확인한 뒤 전체 지점으로 확장한다. 마지막에는 250 m, 500 m, 1 km, 1.5 km의 분포를 비교하고 선택한 반경의 지점별 표를 저장한 뒤, 사고 한 건마다 해당 지점의 시설 수를 붙인다.

입력은 [01 · 사고 데이터 전처리](01_prepare_crash_data.ipynb)에서 저장한 사고 단위 CSV, 시설 원본, CCTV 운영·위치 자료다.

CCTV만 따라가려면 **3단계의 원본 내려받기 → 3.3절의 종류·좌표·중복 확인 → 6단계의 거리 집계 → 8·9단계의 CSV 저장 → 10단계의 검산** 순서로 읽는다.

## 1. 집계 반경 정하기

`RADIUS_M`은 CSV에 저장할 반경이며 기본값은 1,000 m다. `COMPARE_RADII_M`에는 함께 살펴볼 반경을 미터 단위로 넣는다. 값을 바꾼 뒤 위에서부터 다시 실행한다. 이 노트북은 pandas와 NumPy를 사용한다. 3단계의 원본 엑셀 읽기에는 openpyxl, 10단계의 거리 검산에는 scikit-learn이 필요하다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RADIUS_M = 1000
COMPARE_RADII_M = [250, 500, 1000, 1500]
radii = sorted(set([RADIUS_M, *COMPARE_RADII_M]))
assert all(isinstance(r, int) and r > 0 for r in radii), "반경은 양의 정수(m)로 입력하세요."

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/raw/facilities").is_dir())
raw = root / "data/raw/facilities"
processed = root / "data/processed"
print("비교할 반경(m)", radii, " / 저장할 반경(m)", RADIUS_M)

비교할 반경(m) [250, 500, 1000, 1500]  / 저장할 반경(m) 1000


## 2. 사고 1건과 사고다발 지점 1곳 구분하기

분석용 CSV에서는 같은 지점이 사고 건수만큼 반복된다. 자치구·지점명·중심 좌표로 사고 행을 묶고, 2025년 지점목록의 지점ID와 연결한다. 양쪽의 지점과 지점별 사고 건수가 모두 일치하는지 확인한다. 좌표의 원문은 나중에 분석용 CSV와 다시 연결할 수 있게 보존한다.

In [2]:
site_key = ["자치구", "지점명", "지점위도", "지점경도"]
site_list = pd.read_csv(root / "data/raw/tmacs/2025/지점목록.csv", dtype="string")
accidents = pd.read_csv(processed / "사고_분석용.csv", dtype="string", keep_default_na=False)
accident_counts = accidents.groupby(site_key, dropna=False).size().rename("분석용_사고건수").reset_index()

sites = site_list[["지점ID", *site_key, "지점사고건수"]].merge(
    accident_counts, on=site_key, how="outer", validate="one_to_one", indicator=True
)
assert accidents[site_key].ne("").all().all()
assert sites["_merge"].eq("both").all() and sites["지점ID"].is_unique
assert sites[["지점ID", *site_key]].notna().all().all()
assert sites["지점사고건수"].astype(int).eq(sites["분석용_사고건수"]).all()
sites = sites.drop(columns=["_merge", "지점사고건수"]).rename(columns={"분석용_사고건수": "사고건수"})
sites = sites.sort_values("지점ID").reset_index(drop=True)
print(f"사고 {len(accidents):,}건 → {sites['자치구'].nunique()}개 자치구, 중심점 {len(sites):,}곳")
display(sites.head(3))

사고 4,564건 → 25개 자치구, 중심점 327곳


,지점ID,자치구,지점명,지점위도,지점경도,사고건수
0,202500111110_001n,종로구,동묘앞역교차로,37.57303151320484,127.01564582081477,31
1,202500111110_002n,종로구,흥인지문사거리,37.57132137973151,127.00901469251042,12
2,202500111110_003n,종로구,종로5가역교차로,37.57098078794782,127.00193419703086,18


## 3. 시설 파일과 원본 열 살펴보기

식별번호의 앞자리 0을 보존하도록 처음에는 문자열로 읽는다. 아래 표에서 파일마다 행 수와 열 이름을 확인하고, 아래 시설별 단계에서 어떤 열을 왜 남기는지 살펴본다.

**어느 기관에서 어떤 자료를 받았을까?**

기존 무인단속카메라 CSV는 프로젝트에서 이미 사용하던 자료다. 여기에 보완한 자료는 아래 네 파일이다. 내려받은 파일은 이름만 바꿔 `data/raw/facilities/cctv_originals`에 보관했다.

| 원본 파일 | 공개 자료에서 선택한 항목 | 쓰임 |
|---|---|---|
| `parking_20261001.csv` | 2026-10-01 갱신본의 전체 CSV. [서울시 주정차·전용차로 단속 CCTV](https://data.seoul.go.kr/dataList/OA-20471/S/1/datasetView.do) | 현장구분에서 불법주정차만 고른다. |
| `roster_20260729.csv` | 2026-07-29 운영목록 CSV. [서울도시고속도로 교통관리센터](https://www.data.go.kr/data/15144351/fileData.do) | 집계할 관제 장비 ID를 정한다. |
| `positions_20260211.xlsx` | 설치위치 파일 목록에서 2026-02-11 엑셀. [서울시설공단 설치위치](https://data.seoul.go.kr/dataList/OA-15273/L/1/datasetView.do) | 같은 ID의 최신 좌표를 찾는다. |
| `positions_20230404.xlsx` | 같은 목록에서 2023-04-04 엑셀. [서울시설공단 설치위치](https://data.seoul.go.kr/dataList/OA-15273/L/1/datasetView.do) | 최신 좌표가 비어 있는 ID의 좌표를 보완한다. |

다음 셀은 보관된 파일을 먼저 읽고, 없는 원본만 공식 주소에서 받는다. 내려받은 파일의 SHA-256 값(파일 내용의 지문)을 비교해 항상 같은 판본을 사용한다. 서울시가 다운로드 주소의 내용을 갱신하면 이 프로젝트에 보관한 원본으로 재현할 수 있다.

In [3]:
from hashlib import sha256
from urllib.request import urlopen

originals = raw / "cctv_originals"
originals.mkdir(exist_ok=True)
location_url = "https://datafile.seoul.go.kr/bigfile/iot/inf/nio_download.do?infId=OA-15273&infSeq=1&useCache=false"
download_urls = {
    "parking_20261001.csv": "https://datafile.seoul.go.kr/bigfile/iot/sheet/csv/download.do?infId=OA-20471&srvType=S&serviceKind=1&pageNo=1&ssUserId=SAMPLE_VIEW",
    "roster_20260729.csv": "https://www.data.go.kr/cmm/cmm/fileDownload.do?atchFileId=FILE_000000003688288&fileDetailSn=1",
    "positions_20260211.xlsx": location_url + "&seq=5",
    "positions_20230404.xlsx": location_url + "&seq=3",
}
original_hashes = {
    "parking_20261001.csv": "48f61ffd95bdf52a1abd4a3daa7733503032fc7bb5ccc0ac9d10f425c3e22d2a",
    "roster_20260729.csv": "9d1a89b1d3d6ae2824dc48a232187c414e47020fee1a7877e08b298b6ba8065f",
    "positions_20260211.xlsx": "e3513ed3f9e8e58dd23832b31b37c4336ba08977495156170b14112076b91af5",
    "positions_20230404.xlsx": "95dcce5a745cd6d2704feff949363029fa5dc17600ef076bca73560e682c33f2",
}
for name, url in download_urls.items():
    path = originals / name
    if path.is_file():
        content = path.read_bytes()
    else:
        with urlopen(url, timeout=30) as response:
            content = response.read()
    if sha256(content).hexdigest() != original_hashes[name]:
        raise ValueError(f"{name}의 판본이 다릅니다. 프로젝트에 보관된 원본을 확인하세요.")
    if not path.is_file():
        path.write_bytes(content)
    print(name, "· 원본 확인")

parking_20261001.csv · 원본 확인


roster_20260729.csv · 원본 확인
positions_20260211.xlsx · 원본 확인
positions_20230404.xlsx · 원본 확인


**공식 파일을 분석용 CSV로 읽기**

아래에서 입력 파일명을 한 번 정한다. 주정차·운영목록 CSV는 원본 한글 인코딩(CP949)으로 읽는다. 설치위치 엑셀은 첫 시트에서 장비 ID, 지점명, 경도(XPOINT), 위도(YPOINT)만 선택하고 장비 ID가 비어 있는 행을 제외한다.

In [4]:
files = {
    "신호등": "서울_신호등_위경도.csv",
    "어린이보호구역": "서울_어린이보호구역.csv",
    "무인교통단속카메라": "서울_무인교통단속카메라_2023-2025.csv",
    "서울시 주정차·전용차로": "parking_enforcement_seoul_20261001.csv",
    "교통관리센터 운영목록": "traffic_center_roster_20260729.csv",
    "관제 위치 2026": "traffic_cctv_positions_20260211.csv",
    "관제 위치 2023": "traffic_cctv_positions_20230404.csv",
    "대로변횡단보도": "서울_대로변횡단보도_위경도정리_전체.csv",
    "과속방지턱": "서울시_험프_25개구_위경도_20250417.csv",
}

In [5]:
import warnings

source_tables = {
    "서울시 주정차·전용차로": pd.read_csv(originals / "parking_20261001.csv", encoding="cp949", dtype="string", keep_default_na=False),
    "교통관리센터 운영목록": pd.read_csv(originals / "roster_20260729.csv", encoding="cp949", dtype="string", keep_default_na=False),
}
position_columns = ["CCTV_ID", "LOCATION", "XPOINT", "YPOINT"]
for year, date in [(2026, "20260211"), (2023, "20230404")]:
    # 원본에 표 서식이 없어 나오는 경고는 좌표값을 읽는 데 영향을 주지 않는다.
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Workbook contains no default style", category=UserWarning)
        table = pd.read_excel(originals / f"positions_{date}.xlsx", sheet_name=0, usecols=position_columns,
                              dtype="string", keep_default_na=False, engine="openpyxl")
    source_tables[f"관제 위치 {year}"] = table.loc[table["CCTV_ID"].str.strip().ne("")].reset_index(drop=True)

분석용 CSV가 없으면 UTF-8로 저장하고, 있으면 공식 원본과 대조한다. 엑셀에서 `398`과 `398.0`처럼 좌표 표기가 달라질 수 있어 X·Y 열은 숫자로 비교한다. 장비 ID와 지점명은 문자열 그대로 비교한다. 그다음 셀에서 다른 시설 자료도 함께 읽는다.

In [6]:
for name, original_table in source_tables.items():
    csv_path = raw / files[name]
    if not csv_path.is_file():
        original_table.to_csv(csv_path, index=False, encoding="utf-8-sig")
    stored = pd.read_csv(csv_path, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    expected = original_table.copy()
    for col in ["XPOINT", "YPOINT"]:
        if col in expected:
            expected[col] = pd.to_numeric(expected[col].replace("", pd.NA), errors="raise")
            stored[col] = pd.to_numeric(stored[col].replace("", pd.NA), errors="raise")
    pd.testing.assert_frame_equal(stored, expected, check_exact=False, rtol=0, atol=1e-12)
    print(f"{csv_path.name} · {len(stored):,}행 원본과 일치")

parking_enforcement_seoul_20261001.csv · 4,684행 원본과 일치
traffic_center_roster_20260729.csv · 168행 원본과 일치
traffic_cctv_positions_20260211.csv · 1,326행 원본과 일치
traffic_cctv_positions_20230404.csv · 167행 원본과 일치


In [7]:
source = {name: pd.read_csv(raw / file, dtype="string", encoding="utf-8-sig") for name, file in files.items()}
inventory = pd.DataFrame([
    {"시설": name, "원본 행": len(df), "원본 열": len(df.columns), "열 이름": ", ".join(df.columns)}
    for name, df in source.items()
])
with pd.option_context("display.max_colwidth", None):
    display(inventory)

def select_columns(df, columns):
    print("제외하는 열  |", ", ".join(col for col in df if col not in columns))
    selected = df.loc[:, columns].copy()
    display(selected.head(3))
    return selected

,시설,원본 행,원본 열,열 이름
0,신호등,44019,12,"신호등부착대ID, 위도, 경도, 신호등수량, 배면등수량, 신호등종류, 상태, 설치일, 교체일, 부착대방향, 원본X좌표(EPSG5186), 원본Y좌표(EPSG5186)"
1,어린이보호구역,1601,12,"자치구, 대상시설명, 시설종류, 위도, 경도, 도로명주소, 지번주소, CCTV설치여부, CCTV설치대수, 보호구역도로폭, 관할경찰서명, 데이터기준일자"
2,무인교통단속카메라,4473,14,"카메라ID, 자치구, 위도, 경도, 설치장소, 도로명주소, 지번주소, 도로종류, 단속구분, 제한속도, 보호구역구분, 설치연도, 데이터기준일자, 제공기관명"
3,서울시 주정차·전용차로,4684,6,"고정형CCTV지번주소, 위도, 경도, 자치구, 단속지점명, 현장구분"
4,교통관리센터 운영목록,168,3,"CCTV 운영관리 대수, CCTV 아이디(ID), 위치"
5,관제 위치 2026,1326,4,"CCTV_ID, LOCATION, XPOINT, YPOINT"
6,관제 위치 2023,167,4,"CCTV_ID, LOCATION, XPOINT, YPOINT"
7,대로변횡단보도,31080,9,"자료유형, 대표위도, 대표경도, 시군구명, 읍면동명, 노드ID, 링크ID, 링크길이_m, 원본WKT"
8,과속방지턱,18384,7,"관리번호, 위도, 경도, 시군구명, 험프유형, 설치일, 출처"


### 3.1 신호등 — 부착대별 신호등수량 합산

한 부착대에 등이 여러 개 달려 있다. 부착대ID와 위치를 남기고, 원본의 `신호등수량`을 더한다. `배면등수량`은 별도 항목이므로 합계에서 제외한다. 상태 코드나 설치일로 행을 걸러내지 않아 집계값은 파일에 기록된 수량을 뜻한다.

In [8]:
signals = select_columns(source["신호등"], ["신호등부착대ID", "위도", "경도", "신호등수량"])
display(signals["신호등수량"].value_counts(dropna=False).rename("원본 행 수").to_frame())
repeated_mounts = signals[signals.duplicated("신호등부착대ID", keep=False)]
display(repeated_mounts.sort_values("신호등부착대ID"))

제외하는 열  | 배면등수량, 신호등종류, 상태, 설치일, 교체일, 부착대방향, 원본X좌표(EPSG5186), 원본Y좌표(EPSG5186)


,신호등부착대ID,위도,경도,신호등수량
0,03-0000049112,37.57955,126.9525702,1
1,03-0000019718,37.5221242,126.8428921,1
2,03-0000028064,37.4982297,127.1342475,1


,원본 행 수
신호등수량,
1,29572
2,12358
3,1311
4,709
0,60
6,6
12,3


,신호등부착대ID,위도,경도,신호등수량
20794,03-0000022706,37.5930626,127.0364697,1
21991,03-0000022706,37.5933031,127.0364156,1
41536,03-0000060707,37.564844,127.1754238,1
43825,03-0000060707,37.5645984,127.1751142,1


위 표에서 같은 부착대ID가 서로 다른 좌표에 나타난다. 원본만으로 어느 위치가 맞는지 판정하기 어려워 **부착대ID와 좌표의 조합**을 관측 단위로 삼는다. 같은 위치에 있어도 ID가 다른 부착대의 수량은 각각 반영한다.

### 3.2 어린이보호구역 — 대상시설 대표점 세기

이 파일은 학교·유치원 등의 대표 좌표를 제공한다. 동일 좌표를 공유하는 다른 시설이 있어 자치구·시설종류·대상시설명과 좌표를 함께 보존한다. 이름과 종류, 위치가 일치하는 대상시설 한 곳을 1로 센다. 구역 경계가 원과 겹치는 면적은 이 자료로 계산할 수 없다.

In [9]:
zones = select_columns(source["어린이보호구역"], ["자치구", "시설종류", "대상시설명", "위도", "경도"])
print("같은 좌표에 있는 추가 행", zones.duplicated(["위도", "경도"]).sum())
zone_dates = source["어린이보호구역"]["데이터기준일자"].dropna()
print("보호구역 데이터 기준일 범위", zone_dates.min(), "~", zone_dates.max())

제외하는 열  | 도로명주소, 지번주소, CCTV설치여부, CCTV설치대수, 보호구역도로폭, 관할경찰서명, 데이터기준일자


,자치구,시설종류,대상시설명,위도,경도
0,강남구,어린이집,GKL행복어린이집,37.51043221,127.051243
1,강남구,어린이집,강남구청어린이집,37.51796636,127.0470611
2,강남구,초등학교,개원초등학교,37.47219072,127.0996009


같은 좌표에 있는 추가 행 148
보호구역 데이터 기준일 범위 2019-09-05 ~ 2026-09-21


### 3.3 카메라 원본에서 용도 구분하기

기존 파일의 단속구분 4는 불법주정차다. 기존 자료에 어느 구가 들어 있는지 확인한 뒤, 주정차 부분은 25개 구와 서울시 운영 지점을 함께 제공하는 서울시 자료로 교체한다. 기존 파일에서는 속도·신호·기타 단속 장비를 ID당 한 대씩 센다. 교통관리센터의 소통 관제 CCTV는 별도 항목이다.

In [10]:
camera_source = source["무인교통단속카메라"]
assert camera_source["단속구분"].isin(["1", "2", "4", "99"]).all()
display(pd.crosstab(camera_source["단속구분"], camera_source["제공기관명"].eq("경찰청").map({True: "경찰청", False: "자치구"})))
legacy_parking = camera_source.loc[camera_source["단속구분"].eq("4")].copy()
other_cameras = select_columns(camera_source.loc[camera_source["단속구분"].ne("4")], ["카메라ID", "위도", "경도"])
print(f"기존 주정차 자료는 {legacy_parking['자치구'].nunique()}개 구, {len(legacy_parking):,}행")
print("기존 파일의 기준일 범위", camera_source["데이터기준일자"].min(), "~", camera_source["데이터기준일자"].max())

제공기관명,경찰청,자치구
단속구분,,
1,612,0
2,1852,0
4,0,1925
99,84,0


제외하는 열  | 자치구, 설치장소, 도로명주소, 지번주소, 도로종류, 단속구분, 제한속도, 보호구역구분, 설치연도, 데이터기준일자, 제공기관명


,카메라ID,위도,경도
0,1320000_G1965,37.4828371,127.0525071
1,1320000_G2636,37.4650254,127.0957193
2,1320000_G3519,37.5195631,127.0282931


기존 주정차 자료는 13개 구, 1,925행
기존 파일의 기준일 범위 2023-08-18 ~ 2026-10-20


**서울시 주정차 단속지점 고르기**  
서울시 자료의 ‘자치구’ 열에는 운영 주체인 서울시도 들어 있다. 불법주정차구역만 선택하고 버스·자전거 전용차로는 제외한다. ‘서울시’ 행도 유효한 좌표가 있으면 해당 위치에서 센다. [서울시 주정차·전용차로 단속 CCTV 위치정보](https://data.seoul.go.kr/dataList/OA-20471/S/1/datasetView.do)

In [11]:
parking_source = source["서울시 주정차·전용차로"]
display(parking_source["현장구분"].value_counts().rename("원본 행"))
parking = parking_source.loc[parking_source["현장구분"].eq("불법주정차구역")].copy()
parking_key = ["자치구", "고정형CCTV지번주소", "단속지점명"]
for col in parking_key:
    parking[col] = parking[col].str.strip().str.replace(r"\s+", " ", regex=True)
assert parking[parking_key].notna().all().all() and parking[parking_key].ne("").all().all()
assert set(parking["자치구"]) - {"서울시"} == set(sites["자치구"])
coverage = pd.concat([legacy_parking["자치구"].value_counts().rename("기존 주정차"),
                      parking["자치구"].value_counts().rename("서울시 주정차 원본")], axis=1)
display(coverage.fillna(0).astype(int).sort_index())

현장구분
불법주정차구역    4623
버스전용차로       47
자전거전용차로      14
Name: 원본 행, dtype: int64[pyarrow]

,기존 주정차,서울시 주정차 원본
자치구,,
강남구,363,359
강동구,70,127
강북구,47,118
강서구,0,161
관악구,224,235
광진구,104,110
구로구,0,176
금천구,117,117
노원구,0,161


**서울 주변 좌표와 같은 지점의 중복 확인하기**  
위·경도가 서울 주변 점검 범위에 들어오는지 확인한다. 서울시 자료에는 장비 ID가 없어서 운영 주체·주소·단속지점명이 같은 행을 대조한다. 좌표가 서로 5 m 이내면 같은 지점의 첫 좌표를 한 번 사용한다. 같은 이름과 주소가 5 m를 넘게 떨어져 있으면 위치를 확정하기 어려워 해당 지점을 제외한다. 이름이나 방향이 다른 단속지점은 같은 좌표에 있어도 각각 센다.

In [12]:
def seoul_coordinates_valid(latitude, longitude):
    return latitude.between(37.4, 37.75) & longitude.between(126.7, 127.25)

def distance_m(lat1, lon1, lat2, lon2):
    """위도·경도(도) 두 쌍 사이의 구면 거리(m). 두 번째 점에는 배열을 넣을 수 있다."""
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6_371_008.8 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

In [13]:
coords = ["위도", "경도"]
parking[coords] = parking[coords].apply(pd.to_numeric, errors="coerce")
valid = seoul_coordinates_valid(parking["위도"], parking["경도"]).fillna(False)
print("서울 주변 좌표 범위를 벗어나거나 좌표가 없는 행")
display(parking.loc[~valid, [*parking_key, *coords]])
parking = parking.loc[valid].copy()
repeated = parking.loc[parking.duplicated(parking_key, keep=False)]
exclude = []
for _, group in repeated.groupby(parking_key):
    # 모든 좌표 쌍이 5 m 안에 있는지 확인한다.
    lat, lon = group["위도"].to_numpy(float), group["경도"].to_numpy(float)
    spread = distance_m(lat[:, None], lon[:, None], lat, lon).max()
    print(group.iloc[0]["단속지점명"], f"· 최대 {spread:.1f} m",
          "· 첫 위치 사용" if spread <= 5 else "· 위치 불확실로 제외")
    exclude.extend(group.index[1:] if spread <= 5 else group.index)
parking = parking.drop(index=exclude)
assert not parking.duplicated(parking_key).any()
assert set(parking["자치구"]) - {"서울시"} == set(sites["자치구"])
print(f"집계할 불법주정차 단속지점 {len(parking):,}곳")

display(pd.Series({
    "전체 원본": len(parking_source),
    "불법주정차만 선택": int(parking_source["현장구분"].eq("불법주정차구역").sum()),
    "서울 주변 좌표 확인 후": int(valid.sum()),
    "중복·위치 불확실 처리 후": len(parking),
}, name="남은 행 수").to_frame())

서울 주변 좌표 범위를 벗어나거나 좌표가 없는 행


,자치구,고정형CCTV지번주소,단속지점명,위도,경도
2214,영등포구,영등포동3가 9-13,신세계백화점 주변,35.517833,126.906425
2218,영등포구,대림동 1080-17,관악농협대림지접구로1교,37.489828,126.389985
2717,성북구,돈암동 1,해오름어린이집 주변,37.36114,127.01341


원당어린이집 옆 (시루봉로5길 38) · 최대 70.5 m · 위치 불확실로 제외
응암동337-39 · 최대 1.6 m · 첫 위치 사용
집계할 불법주정차 단속지점 4,617곳


,남은 행 수
전체 원본,4684
불법주정차만 선택,4623
서울 주변 좌표 확인 후,4620
중복·위치 불확실 처리 후,4617


**교통관리센터 운영 ID에 좌표 붙이기**  
공공데이터포털의 운영목록은 센터 관제 장비를 정하는 기준이다. ‘CCTV 운영관리 대수’ 열은 일련번호이므로 ID 한 개를 한 대로 센다. 서울시설공단 위치표에는 다른 장비도 들어 있어 운영목록에서 출발한다. 2026년 위치표에서 좌표 두 값이 모두 유효하면 사용하고, 비어 있으면 같은 ID의 2023년 공식 좌표 쌍을 사용한다. 당시 위치를 사용한 장비는 표에 따로 표시한다. [교통관리센터 운영목록](https://www.data.go.kr/data/15144351/fileData.do) · [서울시설공단 연도별 CCTV 설치위치](https://data.seoul.go.kr/dataList/OA-15273/L/1/datasetView.do)

In [14]:
roster = source["교통관리센터 운영목록"].rename(columns={"CCTV 아이디(ID)": "카메라ID"})
center = roster.set_index("카메라ID")[["위치"]].copy()
assert center.index.is_unique and center.index.notna().all()
for year in [2026, 2023]:
    locations = source[f"관제 위치 {year}"].set_index("CCTV_ID")
    assert locations.index.is_unique and locations.index.notna().all()
    xy = locations.rename(columns={"YPOINT": "위도", "XPOINT": "경도"})[coords]
    xy = xy.reindex(center.index).apply(pd.to_numeric, errors="coerce")
    valid = seoul_coordinates_valid(xy["위도"], xy["경도"]).fillna(False)
    if year == 2026:
        center[coords] = xy.where(valid, axis=0)
        center.loc[valid, "좌표 기준연도"] = year
    else:
        use = center["위도"].isna() & valid
        center.loc[use, coords] = xy.loc[use, coords]
        center.loc[use, "좌표 기준연도"] = year

assert center[coords].notna().all().all(), "좌표를 확보하지 못한 운영 ID 확인 필요"
center = center.reset_index()
print("운영목록과 연결된 장비", len(center), "대")
display(center["좌표 기준연도"].astype(int).value_counts().rename("대수"))
display(center.loc[center["좌표 기준연도"].eq(2023), ["카메라ID", "위치", *coords]])

운영목록과 연결된 장비

 168 대


좌표 기준연도
2026    163
2023      5
Name: 대수, dtype: int64

,카메라ID,위치,위도,경도
148,CHN1000010,[강변]월드컵대교 북단,37.548299,126.882094
158,CCMG900010,[마곡]양천길(서남물재생센터 삼거리),37.572671,126.828622
159,CCMG900020,[마곡]신방화역,37.567408,126.816722
160,CCMG900030,[마곡]마곡역,37.559925,126.827195
161,CCMG900040,[마곡]양천향교역,37.568338,126.841146


**서로 다른 원본에 같은 카메라가 들어갔는지 확인하기**  
각 행에서 상대 자료의 가장 가까운 위치까지 거리를 구한다. 같은 좌표와 10 m 이내 후보 수는 위치 중복을 살펴보는 값이다. 같은 교차로의 신호 단속과 소통 관제는 별도 기능일 수 있다. 주정차의 두 출처는 합치지 않고 서울시 자료만 사용한다. 관제 장비는 기존 장비와 ID도 대조하고 별도 열에 집계한다. 단속구분 2는 신호 단속이다.

In [15]:
def overlap_counts(left, right):
    left_xy = left[coords].to_numpy(float)
    right_lat, right_lon = right[coords].to_numpy(float).T
    nearest = np.array([distance_m(lat, lon, right_lat, right_lon).min() for lat, lon in left_xy])
    return [len(left), int((nearest <= 0.001).sum()), int((nearest <= 10).sum())]

overlap = pd.DataFrame({
    "기존 주정차 → 서울시 주정차": overlap_counts(legacy_parking, parking),
    "기존 주정차 외 단속 → 서울시 주정차": overlap_counts(other_cameras, parking),
    "센터 관제 → 기존 전체": overlap_counts(center, camera_source),
    "센터 관제 → 서울시 주정차": overlap_counts(center, parking),
}, index=["대조한 행", "좌표 일치(1 mm 이내)", "10 m 이내(좌표 일치 포함)"]).T
display(overlap)
old_ids = camera_source["카메라ID"].str.strip().str.upper()
center_ids = set(center["카메라ID"].str.strip().str.upper())
common_ids = center_ids & (set(old_ids) | set(old_ids.str.split("_", n=1).str[-1]))
print("기존 카메라와 센터의 공통 장비 ID", sorted(common_ids))
assert not common_ids, "공통 ID의 용도와 좌표를 확인한 뒤 집계하세요."

# 센터와 기존 장비가 10 m 안에 함께 있는 경우, ID와 용도를 나란히 확인한다.
old_lat, old_lon = camera_source[coords].to_numpy(float).T
near_pairs = []
for device in center.itertuples(index=False):
    distances = distance_m(device.위도, device.경도, old_lat, old_lon)
    for idx in np.flatnonzero(distances <= 10):
        old = camera_source.iloc[idx]
        near_pairs.append({"센터 ID": device.카메라ID, "센터 위치": device.위치, "기존 ID": old["카메라ID"],
                           "기존 단속구분": old["단속구분"], "기존 설치장소": old["설치장소"], "거리(m)": round(distances[idx], 1)})
display(pd.DataFrame(near_pairs))

,대조한 행,좌표 일치(1 mm 이내),10 m 이내(좌표 일치 포함)
기존 주정차 → 서울시 주정차,1925,70,621
기존 주정차 외 단속 → 서울시 주정차,2548,0,107
센터 관제 → 기존 전체,168,0,1
센터 관제 → 서울시 주정차,168,0,0


기존 카메라와 센터의 공통 장비 ID []


,센터 ID,센터 위치,기존 ID,기존 단속구분,기존 설치장소,거리(m)
0,CCN5000420,[남산]경리단,1320000_J7897,2,"재정관리단앞교차로(이태원동, 남산3호터널→녹사평역교차로)",8.0


### 3.4 대로변횡단보도 — 링크 대표점 세기

먼저 위치를 표현하는 두 종류의 행을 구분한다. `NODE`는 점이고 `LINK`는 선 구간이다. 두 행 유형을 합산하면 끝점과 선을 함께 세게 된다. 여기서는 **고유 링크ID 1개**를 횡단보도 구간 1개로 보고, 대표점이 반경에 들어온 구간을 센다. 실제 횡단보도 하나가 여러 구간으로 나뉘어 있을 수 있다.

In [16]:
crossing_source = source["대로변횡단보도"]
display(crossing_source.groupby("자료유형")[["노드ID", "링크ID"]].count())
display(crossing_source.groupby("자료유형").head(1)[["자료유형", "노드ID", "링크ID", "원본WKT"]])

,노드ID,링크ID
자료유형,,
LINK,0,11562
NODE,19518,0


,자료유형,노드ID,링크ID,원본WKT
0,NODE,129739,<NA>,POINT(126.96900687649133 37.57610201706163)
3,LINK,<NA>,175551,LINESTRING(126.97665069643799 37.5703427299231...


In [17]:
links = crossing_source.loc[crossing_source["자료유형"].eq("LINK")]
crossings = select_columns(links, ["링크ID", "대표위도", "대표경도"]).rename(
    columns={"대표위도": "위도", "대표경도": "경도"}
)
print(f"원본 {len(crossing_source):,}행 중 링크 {len(crossings):,}행 사용")

제외하는 열  | 자료유형, 시군구명, 읍면동명, 노드ID, 링크길이_m, 원본WKT


,링크ID,대표위도,대표경도
3,175551,37.570254717,126.97665673
5,80008,37.575015467,127.005633159
6,80080,37.574543102,126.990949973


원본 31,080행 중 링크 11,562행 사용


### 3.5 과속방지턱 — 서울시 25개 구의 험프 관리번호 세기

서울시가 2025년 4월 17일 공개한 험프 자료에서 만든 위경도 CSV를 사용한다. 좌표는 각 도형을 감싸는 사각형의 중심이다. 아래 표에서 자치구별 요철형과 이미지형(노면에 그린 방지턱)의 수를 확인한다. 두 유형 모두 관리번호 하나를 방지턱 1개로 센다. [서울시 험프 관련 정보와 테이블 정의서](https://data.seoul.go.kr/dataList/OA-15541/F/1/datasetView.do)

In [18]:
bump_source = source["과속방지턱"]
bump_districts = set(bump_source["시군구명"].dropna())
assert set(sites["자치구"]) == bump_districts, "사고지점과 과속방지턱 자료의 자치구 목록을 확인하세요."
display(pd.crosstab(bump_source["시군구명"], bump_source["험프유형"], margins=True, margins_name="합계"))

험프유형,요철,이미지,합계
시군구명,,,
강남구,292,598,890
강동구,138,544,682
강북구,147,766,913
강서구,148,534,682
관악구,69,158,227
광진구,122,339,461
구로구,192,818,1010
금천구,105,254,359
노원구,116,780,896


이제 거리 계산에 쓸 관리번호·위도·경도만 남긴다. 각 지점에서는 구 경계를 넘어도 반경 안에 있는 방지턱을 모두 센다.

In [19]:
bumps = select_columns(bump_source, ["관리번호", "위도", "경도"])
print(f"{len(bump_districts)}개 자치구 · 고유 관리번호 {bumps['관리번호'].nunique():,}개")

제외하는 열  | 시군구명, 험프유형, 설치일, 출처


,관리번호,위도,경도
0,16-0000000009,37.515499583,127.032624240
1,16-0000000066,37.513091728,127.030401406
2,16-0000001287,37.515327887,127.032513339


25개 자치구 · 고유 관리번호 18,384개


## 4. 좌표·중복·수량 점검하기

위도와 경도를 숫자로 바꾼다. 비어 있거나 서울 주변의 넉넉한 점검 범위(위도 37.4~37.75°, 경도 126.7~127.25°)를 벗어난 행은 집계에서 빼고 수를 기록한다. 이 사각형은 뒤바뀐 좌표 등을 찾기 위한 범위다.

시설별로 선택한 열이 모두 같은 행은 한 번만 남긴다. 그 뒤에도 같은 식별 기준에 서로 다른 위치나 수량이 남는다면 확인할 수 있도록 오류를 표시한다. 마지막 표의 **집계 수량**은 신호등에서는 등의 합계, 나머지 시설에서는 정리 후 남은 개수다.

In [20]:
assert seoul_coordinates_valid(sites["지점위도"].astype(float), sites["지점경도"].astype(float)).all()

tables = {
    "신호등": (signals, ["신호등부착대ID", "위도", "경도"]),
    "어린이보호구역": (zones, ["자치구", "시설종류", "대상시설명", "위도", "경도"]),
    "주정차 외 무인단속": (other_cameras, ["카메라ID"]),
    "불법주정차CCTV": (parking[[*parking_key, *coords]], parking_key),
    "교통관리센터CCTV": (center[["카메라ID", *coords]], ["카메라ID"]),
    "대로변횡단보도": (crossings, ["링크ID"]),
    "과속방지턱": (bumps, ["관리번호"]),
}
clean = {}
audit = []
for name, (table, key) in tables.items():
    data = table.copy()
    for col in ["위도", "경도"]:
        data[col] = pd.to_numeric(data[col], errors="coerce")
    valid = seoul_coordinates_valid(data["위도"], data["경도"]).fillna(False)
    located = data.loc[valid]
    unique = located.drop_duplicates().copy()
    assert unique[key].notna().all().all(), f"{name} 식별값 누락"
    assert not unique.duplicated(key).any(), f"{name} 식별값이 같은 행의 위치·수량 확인 필요"

    if name == "신호등":
        quantity = pd.to_numeric(unique["신호등수량"], errors="raise")
        assert quantity.notna().all() and quantity.ge(0).all() and quantity.mod(1).eq(0).all()
        unique["수량"] = quantity.astype(int)
    else:
        unique["수량"] = 1
    clean[name] = unique
    source_name = {"주정차 외 무인단속": "무인교통단속카메라", "불법주정차CCTV": "서울시 주정차·전용차로",
                   "교통관리센터CCTV": "교통관리센터 운영목록"}.get(name, name)
    input_rows = len(source[source_name])
    audit.append([name, input_rows, input_rows - len(table), (~valid).sum(),
                  len(located) - len(unique), len(unique), unique["수량"].sum()])

display(pd.DataFrame(audit, columns=["시설", "원본 행", "3단계 제외", "좌표 제외", "중복 제외", "사용 행", "집계 수량"]))

,시설,원본 행,3단계 제외,좌표 제외,중복 제외,사용 행,집계 수량
0,신호등,44019,0,0,0,44019,61129
1,어린이보호구역,1601,0,0,0,1601,1601
2,주정차 외 무인단속,4473,1925,0,0,2548,2548
3,불법주정차CCTV,4684,67,0,0,4617,4617
4,교통관리센터CCTV,168,0,0,0,168,168
5,대로변횡단보도,31080,19518,0,0,11562,11562
6,과속방지턱,18384,0,0,0,18384,18384


## 5. 한 지점에서 카메라까지의 거리 구하기

먼저 한 지점의 주정차 외 무인단속 장비만 살펴본다. 하버사인 공식으로 지점 중심과 시설 대표 좌표 사이의 지표면 최단거리를 미터로 계산한다. 도로를 따라 이동하는 거리는 별도 자료가 필요하다. 아래 셀에서 **거리 ≤ 선택 반경**인 카메라에 표시가 되는지 확인한다.

In [21]:
example = sites.iloc[0]
near_cameras = clean["주정차 외 무인단속"][["카메라ID", "위도", "경도"]].copy()
near_cameras["거리_m"] = distance_m(
    float(example["지점위도"]), float(example["지점경도"]),
    near_cameras["위도"].to_numpy(float), near_cameras["경도"].to_numpy(float),
)
near_cameras["반경 안"] = near_cameras["거리_m"] <= RADIUS_M
example_count = int(near_cameras["반경 안"].sum())
print(f"{example['자치구']} {example['지점명']} · {RADIUS_M:,} m 안 주정차 외 무인단속 카메라 {example_count}대")
print("경계와 가까운 카메라 6대 (판정 후 표시 거리만 반올림)")
closest_to_edge = (near_cameras["거리_m"] - RADIUS_M).abs().nsmallest(6).index
display(near_cameras.loc[closest_to_edge].sort_values("거리_m").round({"거리_m": 1}))

종로구 동묘앞역교차로 · 1,000 m 안 주정차 외 무인단속 카메라 15대


경계와 가까운 카메라 6대 (판정 후 표시 거리만 반올림)


,카메라ID,위도,경도,거리_m,반경 안
3997,1320000_G5598,37.5813,127.012945,949.7,True
4009,1320000_H0836,37.581424,127.012452,974.7,True
2964,1320000_G6855,37.580396,127.022102,997.2,True
4284,1320000_J7788,37.56652,127.007787,1002.0,False
4288,1320000_J7872,37.566087,127.008007,1024.5,False
2881,1320000_G3815,37.567679,127.025735,1070.0,False


## 6. 모든 지점에서 시설 수 세기

같은 계산을 모든 지점에 적용한다. 시설과 지점 사이 거리는 한 번 구하고, 작은 반경부터 큰 반경까지 해당 시설의 수량을 합산한다. 시설이 없는 조합은 0으로 채운다. 서로 다른 두 지점의 원이 겹치면 같은 시설이 두 지점에 각각 포함될 수 있다.

In [22]:
facility_names = list(clean)
points = pd.concat([
    table[["위도", "경도", "수량"]].assign(시설=name) for name, table in clean.items()
], ignore_index=True)
facility_lats = points["위도"].to_numpy(float)
facility_lons = points["경도"].to_numpy(float)

rows = []
for site in sites.itertuples(index=False):
    distances = distance_m(float(site.지점위도), float(site.지점경도), facility_lats, facility_lons)
    for radius in radii:
        inside = points.loc[distances <= radius]
        counts = inside.groupby("시설")["수량"].sum().reindex(facility_names, fill_value=0)
        rows.append({"지점ID": site.지점ID, "반경_m": radius, **counts.to_dict()})

by_radius = pd.DataFrame(rows)

**단속 합계와 전체 CCTV 집계수 구하기**

먼저 불법주정차 단속지점 수에 주정차 외 무인단속 장비 수(속도·신호·기타)를 더해 단속 합계를 구한다. 여기에 교통관리센터 관제 장비 수를 더하면 전체 CCTV 집계수다. 주정차 지점은 이미 단속 합계에 들어 있다. 전체 값의 단위는 등록된 지점과 장비를 합한 ‘건’이다.

In [23]:
by_radius["무인교통단속카메라"] = by_radius["불법주정차CCTV"] + by_radius["주정차 외 무인단속"]
by_radius["전체CCTV"] = by_radius["무인교통단속카메라"] + by_radius["교통관리센터CCTV"]
facility_names = [*clean, "무인교통단속카메라", "전체CCTV"]

아래 표는 앞에서 본 한 지점의 결과다. 반경을 넓히면 시설 수가 줄어들지 않는지, 선택 반경의 카메라 수가 5단계에서 직접 센 값과 일치하는지 확인한다.

In [24]:
assert len(by_radius) == len(sites) * len(radii) and not by_radius.duplicated(["지점ID", "반경_m"]).any()
growth = by_radius.groupby("지점ID")[facility_names].diff().dropna()
assert growth.ge(0).all().all(), "반경이 늘었는데 수량이 줄어든 지점 확인 필요"
example_rows = by_radius.loc[by_radius["지점ID"].eq(example["지점ID"])]
assert example_rows.loc[example_rows["반경_m"].eq(RADIUS_M), "주정차 외 무인단속"].item() == example_count
print(example["자치구"], example["지점명"])
display(example_rows)

종로구 동묘앞역교차로


,지점ID,반경_m,신호등,어린이보호구역,주정차 외 무인단속,불법주정차CCTV,교통관리센터CCTV,대로변횡단보도,과속방지턱,무인교통단속카메라,전체CCTV
0,202500111110_001n,250,48,0,1,10,0,11,4,11,11
1,202500111110_001n,500,143,2,2,25,0,40,16,27,27
2,202500111110_001n,1000,626,15,15,92,0,167,72,107,107
3,202500111110_001n,1500,1365,36,40,169,0,329,293,209,209


## 7. 어느 반경이 분석에 적당할까?

아래 두 표는 각 지점을 동등하게 한 번씩 반영한다. 첫 표의 중앙값은 전형적인 지점에서 관측되는 수량을, 둘째 표는 시설이 한 건도 잡히지 않는 지점의 비율을 보여준다. 관심 시설의 0 비율이 높다면 다음 반경에서 얼마나 줄어드는지 확인한다. 중앙값이 급격히 늘어나는 반경에서는 인근의 다른 교차로까지 함께 설명하게 될 수 있다.

In [25]:
print("반경별 지점당 시설 수 중앙값")
display(by_radius.groupby("반경_m")[facility_names].median())
print("반경 안에서 시설이 0개로 집계된 지점 비율 (%)")
no_facility = by_radius[facility_names].eq(0)
display(no_facility.groupby(by_radius["반경_m"]).mean().mul(100).round(1))

반경별 지점당 시설 수 중앙값


,신호등,어린이보호구역,주정차 외 무인단속,불법주정차CCTV,교통관리센터CCTV,대로변횡단보도,과속방지턱,무인교통단속카메라,전체CCTV
반경_m,,,,,,,,,
250,44.0,0.0,1.0,3.0,0.0,9.0,7.0,5.0,5.0
500,124.0,3.0,4.0,11.0,0.0,23.0,32.0,15.0,15.0
1000,464.0,11.0,17.0,37.0,0.0,87.0,134.0,54.0,55.0
1500,984.0,24.0,37.0,74.0,1.0,188.0,279.0,109.0,110.0


반경 안에서 시설이 0개로 집계된 지점 비율 (%)


,신호등,어린이보호구역,주정차 외 무인단속,불법주정차CCTV,교통관리센터CCTV,대로변횡단보도,과속방지턱,무인교통단속카메라,전체CCTV
반경_m,,,,,,,,,
250,0.0,63.9,27.5,8.3,94.5,1.2,5.2,2.4,1.8
500,0.0,12.5,1.8,1.2,85.3,0.0,0.0,0.0,0.0
1000,0.0,0.0,0.0,0.0,59.6,0.0,0.0,0.0,0.0
1500,0.0,0.0,0.0,0.0,38.2,0.0,0.0,0.0,0.0


서울에 있는 시설만 입력했으므로 시 경계 인근 지점에서는 원의 서울 밖 부분이 집계되지 않는다. 시설별 조사 기준일과 설치일 기록이 달라 수량은 **제공 파일 기준의 주변 시설 현황**으로 해석한다. 사고가 발생한 해에 같은 수량이 있었는지는 별도 확인이 필요하다.

## 8. 선택한 반경의 지점별 결과 저장하기

지점ID와 중심 좌표를 살리고, 오해하기 쉬운 집계 단위는 열 이름에 적는다. `과속방지턱_개수`에는 선택한 반경 안에 있는 요철형과 이미지형 방지턱의 합계를 기록한다. 다른 사고유형 분석에서 이 표를 연결할 때는 지점ID를 사용하거나, 지점ID가 없는 분석용 CSV에서는 2단계의 네 열을 사용한다.

단속 합계는 주정차 외 장비 대수와 주정차 단속지점 수의 합이다. 서울시 주정차 자료의 등록 단위가 지점이어서 기존 ‘대수’ 열 이름을 ‘집계수’로 바꿨다. 단속 합계에 센터 관제 대수를 더한 열이 전체CCTV_집계수다.

In [26]:
selected = by_radius.loc[by_radius["반경_m"].eq(RADIUS_M)]
result = sites.merge(selected, on="지점ID", validate="one_to_one")
result = result.rename(columns={
    "신호등": "신호등_수량", "어린이보호구역": "어린이보호구역_대상시설수",
    "무인교통단속카메라": "무인교통단속카메라_집계수", "주정차 외 무인단속": "주정차외_무인단속카메라_대수",
    "불법주정차CCTV": "불법주정차CCTV_지점수", "교통관리센터CCTV": "교통관리센터CCTV_대수", "대로변횡단보도": "대로변횡단보도_링크수",
    "과속방지턱": "과속방지턱_개수", "전체CCTV": "전체CCTV_집계수",
})
assert len(result) == len(sites) and result["지점ID"].is_unique
output = processed / f"사고다발지점_주변시설_{RADIUS_M}m_개수.csv"
result.to_csv(output, index=False, encoding="utf-8-sig")
saved = pd.read_csv(output, dtype="string", encoding="utf-8-sig")
pd.testing.assert_frame_equal(saved, result.astype("string"))
print(f"{output.relative_to(root)} · {len(saved):,}개 지점 저장 및 재확인")
display(result.head(5))

data\processed\사고다발지점_주변시설_1000m_개수.csv · 327개 지점 저장 및 재확인


,지점ID,자치구,지점명,지점위도,지점경도,사고건수,반경_m,신호등_수량,어린이보호구역_대상시설수,주정차외_무인단속카메라_대수,불법주정차CCTV_지점수,교통관리센터CCTV_대수,대로변횡단보도_링크수,과속방지턱_개수,무인교통단속카메라_집계수,전체CCTV_집계수
0,202500111110_001n,종로구,동묘앞역교차로,37.57303151320484,127.01564582081477,31,1000,626,15,15,92,0,167,72,107,107
1,202500111110_002n,종로구,흥인지문사거리,37.57132137973151,127.00901469251042,12,1000,673,11,16,99,0,193,85,115,115
2,202500111110_003n,종로구,종로5가역교차로,37.57098078794782,127.00193419703086,18,1000,757,6,17,89,0,223,81,106,106
3,202500111110_004n,종로구,혜화동로타리,37.58557789426827,127.00124620284728,16,1000,389,10,12,39,0,57,79,51,51
4,202500111110_005n,종로구,원남동사거리,37.575893990362005,126.99737354743372,10,1000,570,6,14,65,0,138,63,79,79


## 9. 사고 한 건마다 주변 시설 수 붙이기

2단계에서 읽은 사고 표와 8단계에서 저장 후 다시 읽은 시설 표를 사용한다. 두 표의 자치구·지점명·지점위도·지점경도가 모두 같은 행을 연결한다. 같은 지점에서 발생한 여러 사고에는 같은 시설 수가 붙는다. 지점별 표의 `사고건수`는 `지점사고건수`로 이름을 바꾼다. 이 값은 해당 지점의 전체 사고 건수이므로 사고 단위 표에서 합산하면 중복된다. 통합 CSV에서는 자치구·지점명·중심 좌표로 지점을 구분하고, 집계 반경은 파일명에 표시한다.

In [27]:
site_features = saved.rename(columns={"사고건수": "지점사고건수"})
combined = accidents.merge(site_features, on=site_key, how="left", validate="many_to_one", indicator=True)

assert combined["_merge"].eq("both").all(), "시설 자료와 연결되지 않은 사고가 있습니다."
combined = combined.drop(columns=["_merge", "지점ID", "반경_m"])
pd.testing.assert_frame_equal(combined[accidents.columns], accidents)
print(f"사고 {len(combined):,}건 연결 · 지점 {len(combined[site_key].drop_duplicates()):,}곳 · 반경 {RADIUS_M:,} m")
display(combined[["지점명", "연도", "사고유형_세부", "신호등_수량", "어린이보호구역_대상시설수",
                  "무인교통단속카메라_집계수", "불법주정차CCTV_지점수", "교통관리센터CCTV_대수", "전체CCTV_집계수", "대로변횡단보도_링크수", "과속방지턱_개수"]].head())

사고 4,564건 연결 · 지점 327곳 · 반경 1,000 m


,지점명,연도,사고유형_세부,신호등_수량,어린이보호구역_대상시설수,무인교통단속카메라_집계수,불법주정차CCTV_지점수,교통관리센터CCTV_대수,전체CCTV_집계수,대로변횡단보도_링크수,과속방지턱_개수
0,수서역앞교차로,2023,차대차-기타,272,6,20,15,3,23,45,54
1,수서역앞교차로,2023,측면직각충돌,272,6,20,15,3,23,45,54
2,수서역앞교차로,2023,추돌(진행중),272,6,20,15,3,23,45,54
3,수서역앞교차로,2023,추돌(진행중),272,6,20,15,3,23,45,54
4,수서역앞교차로,2023,차대차-기타,272,6,20,15,3,23,45,54


In [28]:
combined_path = processed / f"사고_분석용_주변시설_{RADIUS_M}m_CCTV보완.csv"
combined.to_csv(combined_path, index=False, encoding="utf-8-sig")
reloaded = pd.read_csv(combined_path, dtype="string", keep_default_na=False)
pd.testing.assert_frame_equal(reloaded, combined)
print(f"{combined_path.relative_to(root)} · {len(reloaded):,}행 × {len(reloaded.columns)}열")

data\processed\사고_분석용_주변시설_1000m_CCTV보완.csv · 4,564행 × 36열


## 10. 원본 좌표로 지점별·사고별 CCTV 수 검산하기

**① 지점별 CSV를 별도 거리 계산과 대조한다.** 6단계는 지점마다 모든 시설까지의 거리를 직접 구했다. 이번에는 3.3절과 4단계에서 정리한 카메라 좌표로 검색용 자료구조(`BallTree`)를 만들고, 각 지점에서 선택 반경 안에 있는 항목을 다시 센다. 두 계산 모두 위·경도를 라디안으로 바꾸고 지구 반경 6,371,008.8 m를 쓴다. 표의 ‘불일치 지점’이 0이면 저장한 개수가 모든 지점에서 같다는 뜻이다.

In [29]:
from sklearn.neighbors import BallTree

camera_columns = {
    "불법주정차CCTV": "불법주정차CCTV_지점수",
    "주정차 외 무인단속": "주정차외_무인단속카메라_대수",
    "교통관리센터CCTV": "교통관리센터CCTV_대수",
}
# 같은 지점을 같은 순서로 비교하기 위해 지점ID로 맞춘다.
site_rows = sites.set_index("지점ID")
site_xy = np.radians(site_rows[["지점위도", "지점경도"]].to_numpy(float))
saved_counts = saved.set_index("지점ID").loc[site_rows.index]
expected_counts = pd.DataFrame(index=site_rows.index)
checks = []
for facility, col in camera_columns.items():
    camera_xy = np.radians(clean[facility][coords].to_numpy(float))
    tree = BallTree(camera_xy, metric="haversine")
    counts = tree.query_radius(site_xy, r=RADIUS_M / 6_371_008.8, count_only=True)
    expected_counts[col] = counts
    different = int((saved_counts[col].astype(int) != counts).sum())
    checks.append([facility, len(camera_xy), counts.min(), counts.max(), (counts > 0).sum(), different])

check_table = pd.DataFrame(checks, columns=["종류", "원본 정리 후", "지점별 최소", "지점별 최대", "1개 이상인 지점", "불일치 지점"])
display(check_table)
assert check_table["불일치 지점"].eq(0).all()

total_column = "무인교통단속카메라_집계수"
expected_counts[total_column] = expected_counts["불법주정차CCTV_지점수"] + expected_counts["주정차외_무인단속카메라_대수"]
assert expected_counts[total_column].eq(saved_counts[total_column].astype(int)).all()
# 전체는 좌표로 다시 센 세 원자료의 개수를 직접 더해 검산한다.
expected_counts["전체CCTV_집계수"] = expected_counts[list(camera_columns.values())].sum(axis=1)
assert expected_counts["전체CCTV_집계수"].eq(saved_counts["전체CCTV_집계수"].astype(int)).all()
print(f"{len(site_rows)}개 지점의 단속 합계와 전체 CCTV 집계수 일치")

,종류,원본 정리 후,지점별 최소,지점별 최대,1개 이상인 지점,불일치 지점
0,불법주정차CCTV,4617,4,105,327,0
1,주정차 외 무인단속,2548,5,43,327,0
2,교통관리센터CCTV,168,0,7,132,0


327개 지점의 단속 합계와 전체 CCTV 집계수 일치


**② 검산한 지점 수치가 각 사고 행에도 그대로 들어갔는지 확인한다.** 위에서 독립적으로 계산한 다섯 열을 원래 사고 표에 지점 기준으로 붙인다. 디스크에서 다시 읽은 통합 CSV와 사고 행 순서·지점·카메라 수를 비교한다. 마지막에는 중구 충무로역앞교차로 한 곳을 골라 단속 합계, 관제 대수, 전체 CCTV 집계수를 직접 읽어 본다.

In [30]:
verified_sites = site_rows[site_key].join(expected_counts)
expected_events = accidents[site_key].merge(verified_sites, on=site_key, how="left", validate="many_to_one")
pd.testing.assert_frame_equal(reloaded[site_key], expected_events[site_key])
camera_fields = list(expected_counts.columns)
assert expected_events[camera_fields].notna().all().all(), "연결되지 않은 사고 지점이 있습니다."
matches = reloaded[camera_fields].astype(int).eq(expected_events[camera_fields].astype(int)).all(axis=1)
print(f"{RADIUS_M:,} m · 사고 {len(reloaded):,}건 중 {matches.sum():,}건 일치 · 불일치 {(~matches).sum()}건")
assert matches.all()

example_cctv = saved_counts.loc[(saved_counts["자치구"] == "중구") & (saved_counts["지점명"] == "충무로역앞교차로")]
display(example_cctv[["자치구", "지점명", *camera_fields]])
for _, row in example_cctv.iterrows():
    print(f"주정차 {row['불법주정차CCTV_지점수']}곳 + 주정차 외 무인단속 {row['주정차외_무인단속카메라_대수']}대 "
          f"= 단속 합계 {row[total_column]}건. 여기에 관제 {row['교통관리센터CCTV_대수']}대를 더하면 전체 {row['전체CCTV_집계수']}건")

1,000 m · 사고 4,564건 중 4,564건 일치 · 불일치 0건


,자치구,지점명,불법주정차CCTV_지점수,주정차외_무인단속카메라_대수,교통관리센터CCTV_대수,무인교통단속카메라_집계수,전체CCTV_집계수
지점ID,,,,,,,
202500111140_006n,중구,충무로역앞교차로,66,16,2,82,84


주정차 66곳 + 주정차 외 무인단속 16대 = 단속 합계 82건. 여기에 관제 2대를 더하면 전체 84건


### CCTV 보완 전후의 수는 얼마나 달라졌을까?

먼저 서울 전체에서 자료에 수록된 수를 비교한다. 기존 주정차 자료는 13개 구의 장비 ID, 보완 자료는 25개 구와 서울시가 제공한 단속지점을 센다. 주정차 외 장비는 같은 자료를 쓴다. 교통관리센터 CCTV는 이번에 새로 집계한 관제 장비로 단속 합계와 별도로 표시한다. 수록 범위와 단위가 바뀌었으므로 이 차이로 새로 설치된 장비 대수를 알 수는 없다.

In [31]:
old_parking_count = legacy_parking["카메라ID"].nunique()
new_parking_count = len(clean["불법주정차CCTV"])
other_count = len(clean["주정차 외 무인단속"])
city_change = pd.DataFrame({
    "보완 전": [old_parking_count, other_count, old_parking_count + other_count],
    "보완 후": [new_parking_count, other_count, new_parking_count + other_count],
}, index=["불법주정차", "주정차 외 무인단속", "무인단속 합계"])
city_change["차이"] = city_change["보완 후"] - city_change["보완 전"]
display(city_change)
print(f"교통관리센터 CCTV는 기존 자료에서 미집계 → 보완 후 {len(clean['교통관리센터CCTV'])}대 별도 집계")

,보완 전,보완 후,차이
불법주정차,1925,4617,2692
주정차 외 무인단속,2548,2548,0
무인단속 합계,4473,7165,2692


교통관리센터 CCTV는 기존 자료에서 미집계 → 보완 후 168대 별도 집계


다음은 사고지점을 한 곳씩 비교한다. 보완 전 카메라 좌표로 같은 반경의 수를 다시 세고, 보완 후 지점별 CSV의 값에서 그 수를 뺀다. 충무로역앞교차로 예시와 함께 증가·유지·감소한 지점 수를 확인한다. 이미 수록됐던 구도 서울시 자료의 좌표와 단속지점으로 교체됐기 때문에 개별 지점의 수는 줄어들 수도 있다.

In [32]:
old_devices = camera_source[["카메라ID", *coords]].drop_duplicates().copy()
old_devices[coords] = old_devices[coords].apply(pd.to_numeric, errors="coerce")
old_devices = old_devices.loc[seoul_coordinates_valid(old_devices["위도"], old_devices["경도"])]
assert old_devices["카메라ID"].is_unique
old_tree = BallTree(np.radians(old_devices[coords].to_numpy(float)), metric="haversine")
site_change = site_rows[["자치구", "지점명"]].copy()
site_change["보완 전"] = old_tree.query_radius(site_xy, r=RADIUS_M / 6_371_008.8, count_only=True)
site_change["보완 후"] = saved_counts[total_column].astype(int)
site_change["차이"] = site_change["보완 후"] - site_change["보완 전"]

old_file = processed / f"사고다발지점_주변시설_{RADIUS_M}m.csv"
if old_file.is_file():
    old_saved = pd.read_csv(old_file, dtype={"지점ID": "string"}).set_index("지점ID")
    assert old_saved.index.is_unique and set(old_saved.index) == set(site_change.index)
    assert site_change["보완 전"].eq(old_saved["무인교통단속카메라_대수"].reindex(site_change.index)).all()
    print("보완 전 수를 이전 지점별 CSV와 대조해 모두 일치함을 확인")

print(f"반경 {RADIUS_M:,} m · 사고지점 {len(site_change)}곳")
display(site_change[["보완 전", "보완 후"]].agg(["mean", "median"]).rename(index={"mean": "평균", "median": "중앙값"}).round(2))
delta = site_change["차이"]
print(f"증가 {delta.gt(0).sum()}곳 · 유지 {delta.eq(0).sum()}곳 · 감소 {delta.lt(0).sum()}곳")
display(site_change.loc[example_cctv.index])

보완 전 수를 이전 지점별 CSV와 대조해 모두 일치함을 확인
반경 1,000 m · 사고지점 327곳


,보완 전,보완 후
평균,37.53,58.13
중앙값,34.00,54.00


증가 297곳 · 유지 7곳 · 감소 23곳


,자치구,지점명,보완 전,보완 후,차이
지점ID,,,,,
202500111140_006n,중구,충무로역앞교차로,23,82,59


기본 반경 1,000 m로 실행하면 9단계가 사고별 통합 CSV를 저장하고, 10단계가 지점·사고 단위의 카메라 수를 검산한다. [03 · 시설 수에 따른 사고유형 비중 비교](03_compare_crash_type_shares.ipynb)에서 검산한 CSV로 구간별 비중과 히트맵을 만든다.